In [1]:
import sys
from pathlib import Path
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "requirements.txt").exists())
sys.path.insert(0, str(ROOT))

import time
import pandas as pd
from pyspark.sql import functions as F

from src.config import load_config
from src.spark_session import get_spark
from src.evaluation.metrics import evaluate
from src.models.popularity import add_recent_counts, smoothed_ctr
from src.models.content import fit_tfidf, user_profiles, score_content

cfg = load_config("small")
spark = get_spark(cfg)
OUT = cfg["paths"]["processed"]
RES = ROOT / "results" / "metrics"

MIN_DFS = (2, 5)
HIST_KS = (None, 30, 10)        # None = historial completo

In [2]:
news = spark.read.parquet(str(OUT / "news"))
imp = spark.read.parquet(str(OUT / "impressions_long"))
beh = spark.read.parquet(str(OUT / "behaviors"))

# Usuarios de val con su historial (fijo por usuario)
users_val = (beh.filter("periodo = 'val'")
                .select("user_id", "history", "hist_len")
                .dropDuplicates(["user_id"])
                .cache())

# CTR reciente (baseline principal: 24 h, alpha = 20), solo con los datos necesarios para val
p0 = imp.filter("periodo = 'train'").agg(F.avg("label")).first()[0]
base = (imp.filter(F.col("periodo").isin("train", "val"))
           .filter(F.col("time") >= F.to_timestamp(F.lit("2019-11-13 00:00:00")))
           .select("impression_id", "user_id", "time", "news_id", "label", "periodo"))

cands = (add_recent_counts(base, hours=(24,))
         .filter("periodo = 'val'")
         .withColumn("ctr", smoothed_ctr(24, 20, p0))
         .join(F.broadcast(users_val.select("user_id", "hist_len")), "user_id")
         .withColumn("con_hist", F.col("hist_len") > 0)
         .select("impression_id", "user_id", "news_id", "label", "ctr", "con_hist")
         .cache())

cands.groupBy("con_hist").agg(F.countDistinct("impression_id").alias("impresiones"),
                              F.count("*").alias("candidatas")).show()

+--------+-----------+----------+
|con_hist|impresiones|candidatas|
+--------+-----------+----------+
|    true|      29498|   1191854|
|   false|        772|     30575|
+--------+-----------+----------+



In [3]:
def eval_score(df, score_col, nombre, conjunto):
    per_imp, res = evaluate(df.withColumn("score", F.col(score_col)))
    per_imp.unpersist()
    return {"modelo": nombre, "conjunto": conjunto, **res}

filas = [
    eval_score(cands, "ctr", "baseline_ctr_24h_a20", "todas"),
    eval_score(cands.filter("con_hist"), "ctr", "baseline_ctr_24h_a20", "con_historial"),
]
pd.DataFrame(filas)[["modelo", "conjunto", "auc", "mrr", "ndcg5", "ndcg10", "impresiones"]].round(4)

,modelo,conjunto,auc,mrr,ndcg5,ndcg10,impresiones
0,baseline_ctr_24h_a20,todas,0.6475,0.2786,0.2936,0.3567,30270
1,baseline_ctr_24h_a20,con_historial,0.6479,0.2789,0.2940,0.3571,29498


In [4]:
for min_df in MIN_DFS:
    t0 = time.time()
    model, vectorizer = fit_tfidf(news, min_df=min_df)
    news_vec = model.transform(news).select("news_id", "tokens", "vec").cache()
    vocab = len(model.stages[2].vocabulary)
    print(f"\nminDF = {min_df}: vocabulario de {vocab:,} términos ({time.time() - t0:.0f} s)")

    for k in HIST_KS:
        t0 = time.time()
        hist_txt = "completo" if k is None else f"ultimas{k}"
        nombre = f"contenido_mindf{min_df}_{hist_txt}"

        profiles = user_profiles(users_val, news_vec, vectorizer, k=k).cache()
        scored = (score_content(cands, news_vec, profiles)
                  .withColumn("hibrido", F.when(F.col("con_hist"), F.col("cos")).otherwise(F.col("ctr"))))

        # Variante 1: contenido puro, solo usuarios con historial
        filas.append(eval_score(scored.filter("con_hist"), "cos", nombre, "con_historial"))
        # Variante 2: híbrido con respaldo de popularidad, todas las impresiones
        filas.append(eval_score(scored, "hibrido", nombre.replace("contenido", "hibrido"), "todas"))

        print(f"  {hist_txt:9s} AUC con_hist = {filas[-2]['auc']:.4f} | "
              f"AUC híbrido (todas) = {filas[-1]['auc']:.4f} ({time.time() - t0:.0f} s)")
        profiles.unpersist()

    news_vec.unpersist()


minDF = 2: vocabulario de 36,660 términos (9 s)
  completo  AUC con_hist = 0.5897 | AUC híbrido (todas) = 0.5907 (352 s)
  ultimas30 AUC con_hist = 0.5841 | AUC híbrido (todas) = 0.5853 (327 s)
  ultimas10 AUC con_hist = 0.5664 | AUC híbrido (todas) = 0.5680 (307 s)

minDF = 5: vocabulario de 21,309 términos (13 s)
  completo  AUC con_hist = 0.5905 | AUC híbrido (todas) = 0.5916 (642 s)
  ultimas30 AUC con_hist = 0.5850 | AUC híbrido (todas) = 0.5862 (408 s)
  ultimas10 AUC con_hist = 0.5667 | AUC híbrido (todas) = 0.5684 (293 s)


In [5]:
tabla = (pd.DataFrame(filas)[["modelo", "conjunto", "auc", "mrr", "ndcg5", "ndcg10", "impresiones"]]
         .sort_values(["conjunto", "auc"], ascending=[True, False])
         .reset_index(drop=True))

(tabla.assign(periodo="val", dataset=cfg["dataset"])
      .to_csv(RES / "04_contenido_val.csv", index=False))
tabla.round(4)

,modelo,conjunto,auc,mrr,ndcg5,ndcg10,impresiones
0,baseline_ctr_24h_a20,con_historial,0.6479,0.2789,0.2940,0.3571,29498
1,contenido_mindf5_completo,con_historial,0.5905,0.2783,0.2941,0.3515,29498
2,contenido_mindf2_completo,con_historial,0.5897,0.2779,0.2935,0.3509,29498
3,contenido_mindf5_ultimas30,con_historial,0.5850,0.2751,0.2897,0.3471,29498
4,contenido_mindf2_ultimas30,con_historial,0.5841,0.2745,0.2892,0.3466,29498
5,contenido_mindf5_ultimas10,con_historial,0.5667,0.2630,0.2744,0.3324,29498
6,contenido_mindf2_ultimas10,con_historial,0.5664,0.2627,0.2743,0.3320,29498
7,baseline_ctr_24h_a20,todas,0.6475,0.2786,0.2936,0.3567,30270
8,hibrido_mindf5_completo,todas,0.5916,0.2782,0.2937,0.3514,30270
9,hibrido_mindf2_completo,todas,0.5907,0.2778,0.2932,0.3508,30270


In [6]:
# Vuelve a construir el mejor vocabulario para mirar un perfil concreto
model, vectorizer = fit_tfidf(news, min_df=2)
vocab = model.stages[2].vocabulary
news_vec = model.transform(news).select("news_id", "tokens", "vec")
perfiles = user_profiles(users_val.filter("hist_len >= 10").limit(200), news_vec, vectorizer, k=30)

u = perfiles.first()
top = sorted(zip(u["u_vec"].indices, u["u_vec"].values), key=lambda x: -x[1])[:15]
print(f"Usuario {u['user_id']} — términos con más peso en su perfil:")
print(", ".join(f"{vocab[i]} ({w:.2f})" for i, w in top))

Usuario U10272 — términos con más peso en su perfil:
heat (0.17), hikers (0.14), waiters (0.11), kevin (0.11), gronkowski (0.11), patrick (0.10), drexler (0.10), spacey (0.10), nba (0.10), olajuwon (0.10), zo (0.10), pencil (0.10), rob (0.09), kravitz (0.09), mcgowan (0.09)


In [7]:
spark.stop()